In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.14 Wave Optics: Diffraction, Interference, and the Fourier Lens

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.14",
    title="Wave Optics: Diffraction, Interference, and the Fourier Lens",
    blurb="Light behaving as the wave Maxwell said it is: Young's fringes "
    "and the single-slit envelope computed by FFT and matched to their "
    "closed forms, the Airy disk and the resolution limit of every "
    "telescope, the Fresnel number's arc from shadow to far field, the "
    "quarter-intensity knife edge, the bright spot at the center of a "
    "disk's shadow that decided the wave theory in 1818, and a lens "
    "revealed as an analog Fourier transformer.",
    difficulty="intermediate",
    estimate="120–150 min",
)

## Notebook overview

[§3.8](maxwell-waves.ipynb) established that light is an
electromagnetic wave; this notebook computes what waves *do* when
they meet obstacles — which is the subject that decided, between 1801
and 1818, that light is a wave at all. The punchline that organizes
everything: **diffraction is a Fourier transform**. The far field of
an aperture is the Fourier transform of the aperture; a lens performs
that transform optically, in analog, at the speed of light; and the
FFT of [§0.6](../00-foundations/fft.ipynb) is therefore the entire
computational machinery of this notebook. One `numpy.fft` call
replaces every diffraction integral we meet.

We verify the machinery against every closed form the subject owns —
the $\mathrm{sinc}^2$ single-slit envelope, Young's $\cos^2$ fringes,
the Airy pattern and Rayleigh's resolution criterion, the
quarter-intensity knife edge — and then spend it on the two showpieces:
the Fresnel number's continuous arc from geometric shadow to
Fraunhofer far field, and the **Arago spot**, the bright point at the
dead center of a circular obstacle's shadow that Poisson derived as a
*reductio ad absurdum* of Fresnel's theory and Arago promptly
observed. Born and Wolf {cite}`bornwolf1999` and Goodman
{cite}`goodman2005` are the standing references.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**Fraunhofer diffraction is a Fourier transform.** Huygens'
principle — every point of a wavefront re-radiates — becomes, for an
aperture $A(x)$ illuminated by a plane wave and observed far away, a
sum of spherical wavelets whose phases are linear in $x$: exactly a
Fourier integral. The far-field amplitude in direction $\sin\theta$
is

```{math}
:label: eq-wo-fraunhofer
U(\theta) \;\propto\; \int A(x)\,
e^{-2\pi i\, x \sin\theta/\lambda}\, dx
\;=\; \tilde A\!\left(\frac{\sin\theta}{\lambda}\right),
```

the aperture's transform evaluated at spatial frequency
$\sin\theta/\lambda$. Every named pattern follows by transforming a
shape: a slit gives $\mathrm{sinc}$, two slits give
$\cos \times \mathrm{sinc}$, a circle gives the Airy function
$2J_1(u)/u$ with its first dark ring at

```{math}
:label: eq-wo-airy
\sin\theta_1 \;=\; 1.22\,\frac{\lambda}{d}
```

— the resolution limit of every telescope, microscope, and camera,
and the reason apertures are made large.

**The angular spectrum: exact propagation at any distance.** Between
the aperture and the far field lives Fresnel diffraction, and the
clean way to compute it is to decompose the field into plane waves
(an FFT), advance each by its own phase, and resum (an inverse FFT):

```{math}
:label: eq-wo-angspec
U(x, z) \;=\; \mathcal F^{-1}\!\left[
\tilde U(f_x, 0)\; e^{\,2\pi i z \sqrt{1/\lambda^2 - f_x^2}}
\right],
```

exact for the scalar wave equation. One subtlety is physics, not
numerics: for $f_x > 1/\lambda$ the square root turns imaginary and
the wave is **evanescent** — it must decay, not propagate. Implement
the root as a complex square root and the decay is automatic; clamp
it to zero (a tempting shortcut) and sharp edges arrive at the screen
haunted by frequencies that should have died in the first wavelength.
These are the same decaying exponentials that tunnel through barriers
in [§6.13](../06-quantum-mechanics/scattering-tunneling.ipynb), and
{eq}`eq-wo-angspec` itself is the free-flight half of
[§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb)'s
split-step propagator with $t \to z$: light and wavefunctions ride
the same mathematics.

**One number rules the regimes.** For an aperture of half-width $w$
observed at distance $z$, the **Fresnel number**

```{math}
:label: eq-wo-fresnel
N_F \;=\; \frac{w^2}{\lambda z}
```

counts the Fresnel zones the aperture exposes. $N_F \gg 1$: geometric
optics, a sharp shadow. $N_F \sim 1$: Fresnel diffraction, edge
ringing and on-axis oscillations. $N_F \ll 1$: the Fraunhofer far
field of {eq}`eq-wo-fraunhofer`. One dial, three centuries of optics.

Units: lengths in units of the wavelength $\lambda = 1$ throughout,
so $\sin\theta$ and spatial frequency are the same number. Everything
is deterministic.

## Setup

Two grids (a long 1D line for slits and edges, a 2D plane for
circular apertures) and two helpers: the Fraunhofer pattern as a
normalized FFT, and the angular-spectrum propagator with the
evanescent decay done right.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from scipy.special import j1

from ecp import validate

LAM = 1.0
N_1D = 2**17
L_1D = 8192.0
DX = L_1D / N_1D
x_1d = (np.arange(N_1D) - N_1D / 2) * DX

N_2D = 2048
L_2D = 512.0
DXY = L_2D / N_2D
xy = (np.arange(N_2D) - N_2D / 2) * DXY
X2, Y2 = np.meshgrid(xy, xy)
R2 = np.hypot(X2, Y2)


def fraunhofer(aperture, dx):
    """Far-field intensity of a 1D aperture, normalized to its peak.

    Implements eq-wo-fraunhofer with one FFT: the pattern is sampled
    at sin θ = λ f_x on the FFT's own frequency grid.

    Parameters
    ----------
    aperture : numpy.ndarray
        Aperture transmission sampled on the working grid.
    dx : float
        Grid spacing.

    Returns
    -------
    tuple of numpy.ndarray
        (sin θ values, normalized intensity).
    """
    F = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(aperture)))
    s = LAM * np.fft.fftshift(np.fft.fftfreq(len(aperture), d=dx))
    I = np.abs(F) ** 2
    return s, I / I.max()


def propagate(u0, dx, z):
    """Angular-spectrum propagation of a 1D field over distance z.

    Implements eq-wo-angspec with the square root taken as a COMPLEX
    root, so spatial frequencies beyond 1/λ decay as the evanescent
    waves they are instead of propagating unphysically.

    Parameters
    ----------
    u0 : numpy.ndarray
        Complex field at z = 0.
    dx : float
        Grid spacing.
    z : float
        Propagation distance.

    Returns
    -------
    numpy.ndarray
        The complex field at distance z.
    """
    fx = np.fft.fftfreq(len(u0), d=dx)
    kz = 2.0 * np.pi * np.sqrt((1.0 / LAM**2 - fx**2).astype(complex))
    return np.fft.ifft(np.fft.fft(u0) * np.exp(1j * kz * z))


def propagate2d(u0, dxy, z):
    """Angular-spectrum propagation of a 2D field over distance z.

    The two-dimensional twin of `propagate`, with the same complex
    square root carrying the evanescent decay.

    Parameters
    ----------
    u0 : numpy.ndarray
        Complex field at z = 0, shape (N, N).
    dxy : float
        Grid spacing.
    z : float
        Propagation distance.

    Returns
    -------
    numpy.ndarray
        The complex field at distance z.
    """
    fx = np.fft.fftfreq(u0.shape[0], d=dxy)
    FX, FY = np.meshgrid(fx, fx)
    kz = 2.0 * np.pi * np.sqrt((1.0 / LAM**2 - FX**2 - FY**2).astype(complex))
    return np.fft.ifft2(np.fft.fft2(u0) * np.exp(1j * kz * z))

## Exercise 1 — Young's experiment, by Fourier transform

The 1801 experiment that started it all, run as {eq}`eq-wo-fraunhofer`
says: transform the aperture.

**Part a)** Single slit of width $a = 8\lambda$. Compute the far
field with `fraunhofer` and verify it against the closed form
$\mathrm{sinc}^2(a_{\rm eff}\sin\theta/\lambda)$ to `atol=1e-2` over
$|\sin\theta| < 0.4$ — where $a_{\rm eff}$ is the width *the sampled
grid actually contains* ($n_{\rm px}\,\Delta x$, one pixel more than
$a$ because both edge samples are inside). Locate the first zero by
minimizing a cubic interpolant of the pattern
(`scipy.optimize.minimize_scalar`) and verify it sits at
$\lambda/a_{\rm eff}$ to `rtol=1e-4` — and within $1\%$ of the
textbook $\lambda/a$, the gap being exactly the pixelization the
lesson of [§0.1](../00-foundations/floating-point.ipynb) predicts:
the FFT diffracts the aperture you *sampled*, not the one you meant.

**Part b)** Two slits of width $a$ separated by $d = 24\lambda$.
Verify the pattern is $\cos^2(\pi d \sin\theta/\lambda)$ fringes
under the $\mathrm{sinc}^2$ envelope (`atol=1e-2`). Then measure the
fringe spacing — carefully, because there is a trap worth a
validation of its own: the *raw* intensity maxima are pulled inward
by the envelope's slope (a product's peak is not the factor's peak),
by about $4\%$ for slits this wide. Verify the pull is there (first
raw peak below $\lambda/d$), then divide the envelope out and verify
the *underlying* fringes sit at exactly $\lambda/d$ spacing
(`rtol=1e-3`). Two numbers — envelope from the slit, fringes from
the pair — and Young could read $\lambda$ off a wall.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    err_single < 1e-2,
    "the FFT of a slit IS the sinc² pattern: Fraunhofer diffraction "
    "as a Fourier transform, to a percent everywhere",
    f"max deviation {err_single:.1e}",
)
validate.close(
    np.array([zero_1]),
    np.array([LAM / a_eff]),
    "the first dark fringe sits at λ/a_eff of the SAMPLED aperture — "
    "within 1% of the textbook λ/a, the gap being one pixel of "
    "aperture width",
    rtol=1e-4,
)
validate.check(
    abs(zero_1 - LAM / A_SLIT) / (LAM / A_SLIT) < 0.01,
    "(and indeed within that 1% of λ/a itself: the FFT diffracts the "
    "aperture you sampled, not the one you meant)",
    f"offset {abs(zero_1 - LAM / A_SLIT) / (LAM / A_SLIT):.4%}",
)
validate.check(
    err_double < 1e-2,
    "two slits give cos² fringes under the same envelope: "
    "interference times diffraction, as the transform of a pair "
    "demands",
    f"max deviation {err_double:.1e}",
)
validate.check(
    0.9 * LAM / D_SEP < first_raw_peak < LAM / D_SEP,
    "the trap, confirmed: the raw intensity maxima are pulled inward "
    "of mλ/d by the envelope's slope — a product's peak is not the "
    "factor's peak",
    f"first raw peak {first_raw_peak:.5f} < λ/d = {LAM / D_SEP:.5f}",
)
validate.close(
    np.array([fringe_spacing]),
    np.array([LAM / D_SEP]),
    "and with the envelope divided out, the fringes sit at exactly "
    "λ/d: the number Young read a wavelength from, two centuries ago",
    rtol=1e-3,
)

## Exercise 2 — The Airy pattern and the resolution of telescopes

In two dimensions the circular aperture rules, and its transform is
the Airy pattern — the point-spread function of every round lens and
mirror ever built.

**Part a)** Transform a circular aperture of diameter $d = 16\lambda$
on the 2D grid (`numpy.fft.fft2`) and verify the axis cut against
the closed form $[2 J_1(u)/u]^2$, $u = \pi d \sin\theta/\lambda$
(`scipy.special.j1`, `atol=5e-3`), with the first dark ring at
{eq}`eq-wo-airy`'s $1.22\,\lambda/d$ to `rtol=5e-3`.

**Part b)** Rayleigh's criterion. Two *incoherent* point sources
(their intensities add) separated by exactly $1.22\,\lambda/d$ —
one source's peak on the other's first dark ring. Verify the summed
profile still shows two peaks with a central dip of $73.5\%$ of the
maximum (`rtol=1e-2`): the conventional edge of "resolved," and the
reason every gain in telescope aperture is a gain in what can be
*seen*.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    err_airy < 5e-3,
    "the 2D FFT of a circle is the Airy pattern, ring for ring",
    f"max deviation {err_airy:.1e}",
)
validate.close(
    np.array([zero_airy]),
    np.array([1.22 * LAM / D_AP]),
    "with the first dark ring at 1.22 λ/d: the diffraction limit of "
    "every round aperture",
    rtol=5e-3,
)
validate.close(
    np.array([dip_ratio]),
    np.array([0.735]),
    "and two incoherent sources at exactly that separation dip to "
    "73.5% between their peaks: Rayleigh's edge of resolved",
    rtol=1e-2,
)

## Exercise 3 — The Fresnel number's arc, and the knife edge

Between the aperture and the far field, {eq}`eq-wo-angspec` propagates
exactly, and {eq}`eq-wo-fresnel` predicts what each distance shows.

**Part a)** Propagate a slab of half-width $w = 20\lambda$ to the
distances where $N_F = 10, 1, 0.1$. Verify the arc: at $N_F = 10$ the
profile is still essentially the geometric shadow (on-axis intensity
within $12\%$ of unity, energy overwhelmingly inside the slab); at
$N_F = 1$ the on-axis intensity has swung to $1.588$
(`rtol=2e-2`) — brighter *behind the middle of the aperture* than
with no aperture at all, pure Fresnel-zone interference; at
$N_F = 0.1$ the light has left geometry behind (on-axis below
$0.5$, the pattern relaxing toward Fraunhofer).

**Part b)** The knife edge — diffraction's cleanest exact numbers.
Propagate a half-plane screen and verify, at two distances: the
intensity at the geometric shadow's edge is exactly $1/4$ of the
incident (`atol=0.005`) — half the amplitude, squared; the first
bright fringe overshoots to $1.370$ (`rtol=1e-2`); and the deep
shadow is dark ($I < 0.05$ three Fresnel scales in). This is the
fringe pattern visible at the edge of every real shadow, and the
$1/4$ requires the evanescent decay of {eq}`eq-wo-angspec` — clamp
the square root to zero, as the Setup warns, and this validation is
the one that catches it.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(arc[10.0][2] - 1.0) < 0.12 and arc[10.0][3] > 0.9,
    "at N_F = 10 geometry still rules: on-axis near unity, the "
    "energy still inside the slab's shadow-to-be",
    f"on-axis {arc[10.0][2]:.3f}, inside fraction {arc[10.0][3]:.3f}",
)
validate.close(
    np.array([arc[1.0][2]]),
    np.array([1.588]),
    "at N_F = 1 the axis is BRIGHTER than with no aperture at all — "
    "Fresnel-zone interference, geometry's first open failure",
    rtol=2e-2,
)
validate.check(
    arc[0.1][2] < 0.5,
    "and at N_F = 0.1 the light has left the aperture's geometry "
    "behind, relaxing toward the Fraunhofer far field",
    f"on-axis {arc[0.1][2]:.3f}",
)
validate.check(
    all(abs(v[1] - 0.25) < 0.005 for v in knife_results.values()),
    "the knife edge shows exactly 1/4 intensity at the geometric "
    "shadow's edge, at both distances: half the amplitude, squared — "
    "and the check that catches a clamped evanescent root",
    f"edges {[round(v[1], 4) for v in knife_results.values()]}",
)
validate.check(
    all(abs(v[2] - 1.370) / 1.370 < 0.01 for v in knife_results.values())
    and all(v[3] < 0.05 for v in knife_results.values()),
    "with the universal 1.37 first overshoot and a genuinely dark "
    "deep shadow: every real shadow's edge, quantified",
    f"overshoots {[round(v[2], 4) for v in knife_results.values()]}",
)

## Exercise 4 — Babinet's principle and the Arago spot

Two of wave optics' most counterintuitive theorems, both one-liners
in the Fourier picture.

**Part a)** Babinet: a screen and its *complement* (opaque where the
screen is open, open where it is opaque) diffract **identically**
everywhere off the forward direction, because their amplitudes sum
to the unobstructed beam — which lives entirely in the forward spike.
Verify on the discrete grid at machine precision: off the axis the
slit's and the anti-slit's far-field intensities agree to
$10^{-10}$, while on axis the complement dominates by more than
$10^4$ (it transmits almost the whole window).

**Part b)** The Arago spot. In 1818 Poisson computed, from Fresnel's
wave memoir, that the exact center of a circular obstacle's shadow
should be *bright* — intended as the theory's death blow. Arago did
the experiment; the spot is there; the wave theory won the prize.
Reproduce the verdict: propagate the field past an opaque disk of
diameter $16\lambda$ with `propagate2d` (adding a soft super-Gaussian
window absorber so the periodic grid edges stay quiet), normalize by
the identically propagated unobstructed beam, and verify the on-axis
intensity
at $z = 100, 200, 400\lambda$ equals the *unobstructed* beam's to
within $5\%$ — every rim point is equidistant from the axis, so the
wavelets arrive in phase, at any distance. The center of the shadow
is as bright as no shadow at all.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    babinet_gap < 1e-10 and axis_ratio > 1e4,
    "Babinet, at machine precision: screen and anti-screen diffract "
    "identically off-axis, differing only in the forward spike that "
    "carries the beam",
    f"off-axis gap {babinet_gap:.1e}, axis ratio {axis_ratio:.1e}",
)
validate.close(
    np.array(list(arago.values())),
    np.ones(3),
    "and Poisson's absurdity is real at all three distances: the "
    "center of the disk's shadow is as bright as no disk at all — "
    "the Arago spot, the wave theory's 1818 verdict",
    rtol=5e-2,
)

## Exercise 5 — The lens as an analog Fourier transformer

A converging lens placed one focal length from an object produces,
one focal length behind it, the object's exact Fourier transform —
in amplitude and phase, in analog, at the speed of light
{cite}`goodman2005`. Put a mask in that focal plane and a second
lens after it (the "$4f$ system"), and you are editing the object's
spectrum by hand: optical information processing, decades before
electronic computers could hold an image.

**Part a)** Simulate the $4f$ system on a bright bar
($120\lambda \times 60\lambda$): forward FFT to the focal plane,
apply a mask, inverse FFT to the image plane. Verify the exact
identities the optics inherits from
[§0.6](../00-foundations/fft.ipynb): an open mask returns the object
to $10^{-12}$; the spectrum's energy splits exactly (Parseval) into
the low-pass and high-pass masks' shares; a pinhole mask passing
only the DC term produces a perfectly uniform image at the object's
mean.

**Part b)** Filter. Verify the low-pass image (frequencies below
$0.02/\lambda$) has its sharpest gradient reduced more than
$5\times$ — edges are high-frequency, so removing high frequencies
*blurs* — while the high-pass image has exactly zero mean (its DC
is in the other mask) and lights up along the object's edges. This
is how a phase-contrast microscope, a spatial filter on a laser
bench, and every "sharpen" button ever shipped all work: they are
masks in a Fourier plane.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    open_gap < 1e-12 and dc_gap < 1e-12,
    "the 4f system with an open mask returns the object exactly, and "
    "a DC pinhole returns its uniform mean: the lens really computes "
    "the transform of §0.6",
    f"open {open_gap:.1e}, DC {dc_gap:.1e}",
)
validate.check(
    parseval_gap < 1e-12,
    "the focal-plane masks split the object's energy exactly "
    "(Parseval): nothing is lost between the optical and the "
    "numerical transform",
    f"split gap {parseval_gap:.1e}",
)
validate.check(
    grad_obj / grad_lp > 5.0 and hp_mean < 1e-12,
    "blocking high frequencies blurs the sharpest edge more than "
    "fivefold, while the high-pass image carries exactly zero mean: "
    "edges LIVE at high spatial frequency",
    f"gradient ratio {grad_obj / grad_lp:.1f}, hp mean {hp_mean:.1e}",
)

## Notebook summary

- One FFT reproduced the $\mathrm{sinc}^2$ single-slit pattern to a
  percent with its first zero at $\lambda/a_{\rm eff}$ to $10^{-4}$
  (and within $1\%$ of the textbook $\lambda/a$ — the gap being one
  pixel of sampled aperture), and Young's $\cos^2$ fringes at
  spacing $\lambda/d$.
- The circular aperture's transform was the Airy pattern with its
  dark ring at $1.22\,\lambda/d$, and two incoherent sources at that
  separation dipped to $73.5\%$: Rayleigh's criterion, measured.
- The Fresnel number walked one slab from geometric shadow
  ($N_F = 10$) through zone interference ($N_F = 1$, on-axis $1.57$)
  toward the far field ($N_F = 0.1$); the knife edge showed exactly
  $1/4$ at the geometric edge and the universal $1.37$ overshoot —
  after the evanescent square root was implemented as the complex
  root it is.
- Babinet held at machine precision off-axis, and the Arago spot —
  Poisson's intended absurdity — sat at the unobstructed intensity
  within $5\%$ at all three distances.
- The $4f$ system computed exact transforms (open mask $10^{-16}$,
  Parseval split exact), and focal-plane masks blurred or extracted
  edges: the lens as the analog ancestor of
  [§0.6](../00-foundations/fft.ipynb).

## Outlook

- **Coherence.** Everything here assumed perfectly coherent light;
  real sources are partially coherent, and fringe *visibility*
  measures it — the van Cittert–Zernike theorem turns that into
  stellar interferometry, sizing stars from fringe contrast
  {cite}`bornwolf1999`.
- **Holography.** Record the focal- or image-plane *phase* (by
  interfering with a reference beam) and the full field can be
  reconstructed: the $4f$ system's masks generalized to recorded
  wavefronts.
- **Adaptive optics.** Atmospheric turbulence corrugates the phase
  that {eq}`eq-wo-angspec` propagates; deformable mirrors flatten it
  in real time, which is why ground telescopes reach the
  $1.22\,\lambda/d$ limit Exercise 2 set.
- **The quantum handshake.** The angular spectrum is
  [§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb)'s
  free propagator and the evanescent tail is
  [§6.13](../06-quantum-mechanics/scattering-tunneling.ipynb)'s
  tunneling wave; single photons sent one at a time through
  Exercise 1's two slits still build the $\cos^2$ pattern — the
  experiment that opens quantum mechanics, with this notebook's
  classical pattern as its envelope.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()